In [1]:
print("Modelo Base")

Modelo Base


In [4]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Cargar los archivos CSV desde tu Drive
train_df = pd.read_csv('/content/drive/MyDrive/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/val.csv')

# Es vital que las etiquetas en el CSV sean strings para que el generador las reconozca
train_df['label'] = train_df['label'].astype(str)
val_df['label'] = val_df['label'].astype(str)

# 2. Configurar el Generador de Datos
# Nota: Usamos preprocess_input de MobileNetV2 en lugar de rescale=1/255
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

# 3. Crear los cargadores de imágenes desde el DataFrame
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='file_path',  # La columna con la ruta completa
    y_col='label',      # La columna con el nombre de la clase
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='file_path',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# 4. Definir la Arquitectura
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    # Usamos el número de clases detectado por el generador
    layers.Dense(len(train_generator.class_indices), activation='softmax')])

# 5. Compilar
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 6. Entrenar
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10 # Puedes subirlo a 20 o más
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/train.csv'